# Lab 1: From Language Models to Agentic Systems

This lab introduces the basic computational building block used throughout the course: a reproducible call to a small language model. We will use the OpenAI **Responses API** and default to `gpt-5-nano`, the lowest-cost GPT-5 model.

We will gradually move from a single model call to a small, observable workflow. The goal is not to build a full agent yet. Instead, we will learn how to identify the pieces that later become an agentic system: a model, instructions, inputs, outputs, state, control logic, tools, observations, and stopping conditions.

**Important:** Never paste an API key directly into this notebook. Use an environment variable or the instructor-managed access method. Model calls consume capped course credit, so run cells deliberately. **Note:** This session is graded, so complete all exercises within class and uploade the saved notebook with outputs to Canvas. 

## References and further reading

1. Roitman, H. (2026). [*The Hitchhiker's Guide to Agentic AI: From Foundations to Systems*](https://arxiv.org/abs/2606.24937), Chapter 15, “Introduction to Agentic AI.” Use this as the primary conceptual reading on the components and autonomy spectrum of agentic systems.
2. OpenAI. [GPT-5 nano model documentation](https://developers.openai.com/api/docs/models/gpt-5-nano). Consult this page for the model's supported endpoints, modalities, context limits, function calling, structured outputs, snapshots, and current rate-limit information.
3. OpenAI. [Text generation with the Responses API](https://developers.openai.com/api/docs/guides/text). This is the primary API reference for the basic response calls, instructions, inputs, and output text used in this notebook.
4. OpenAI. [Structured model outputs](https://developers.openai.com/api/docs/guides/structured-outputs). Read the introductory sections to understand why machine-readable contracts are stronger than merely asking a model to imitate JSON; Week 2 develops this topic fully.
5. OpenAI. [Production best practices](https://developers.openai.com/api/docs/guides/production-best-practices). Focus on API-key safety, staging, rate limits, latency, usage monitoring, and cost controls when moving beyond a classroom experiment.
6. Anthropic. [Building Effective Agents](https://www.anthropic.com/engineering/building-effective-agents). Focus on the distinction between workflows and agents, the augmented-LLM building block, and the guidance to add agentic complexity only when simpler approaches fall short.


## 0. Install and import the required libraries

The notebook uses the official `openai` Python package. We also use `pandas` to display experiment logs. If you are running in Google Colab, execute the installation cell once. After installation, restart the runtime only if Python requests it.

In [ ]:
%pip install -q --upgrade openai pandas

In [ ]:
import os
import json
import time
from datetime import datetime, timezone

import pandas as pd
from openai import OpenAI

MODEL = os.getenv("OPENAI_MODEL", "gpt-5-nano")
print("Model selected:", MODEL)

## 1. Configure API access safely

The OpenAI client automatically reads the `OPENAI_API_KEY` environment variable. The instructor-managed system may provide a compatible base URL and key. If so, follow the instructor's connection instructions.

The cell below checks whether a key is available without printing it. *[Question: Why would printing even the first few characters of a secret be a poor habit in a notebook that may be submitted or shared?]*

In [ ]:
if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(
        "OPENAI_API_KEY is not set. Add it to your environment or use the "
        "instructor-managed access instructions. Do not paste the key into the notebook."
    )

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL") or None,
)
print("Client configured successfully. API key was not displayed.")

## 2. Language model application 1.0: Make a basic model call

A basic language model application contains instructions, an input, a model call, and an output. It is not automatically an agent. The program—not the model—has already decided what operation will happen.

We will ask the model to explain the distinction among a model, a workflow, and an agent in a compact format.

In [ ]:
instructions = (
    "You are a concise teaching assistant for a graduate information-science course. "
    "Use plain language. Do not claim that every multi-step program is an agent."
)

user_input = (
    "Explain the difference among a language model, a deterministic workflow, "
    "and an agent. Give one example of each in no more than 180 words."
)

response = client.responses.create(
    model=MODEL,
    instructions=instructions,
    input=user_input,
)

print(response.output_text)

## 3. Language model application 2.0: Record reproducibility and usage metadata

A screenshot of a good answer is not a reproducible experiment. We should record the model, instructions, input, time, latency, response identifier, output, and token usage when available. This information will later help us compare workflows and debug agents.

In [ ]:
def run_and_log(user_input, instructions, model=MODEL):
    started_at = datetime.now(timezone.utc)
    start_clock = time.perf_counter()

    response = client.responses.create(
        model=model,
        instructions=instructions,
        input=user_input,
    )

    latency_seconds = time.perf_counter() - start_clock
    usage = getattr(response, "usage", None)

    record = {
        "timestamp_utc": started_at.isoformat(),
        "model_requested": model,
        "response_id": getattr(response, "id", None),
        "instructions": instructions,
        "input": user_input,
        "output": response.output_text,
        "latency_seconds": round(latency_seconds, 3),
        "input_tokens": getattr(usage, "input_tokens", None),
        "output_tokens": getattr(usage, "output_tokens", None),
        "total_tokens": getattr(usage, "total_tokens", None),
    }
    return record

experiment_log = []
record = run_and_log(
    user_input="Give three reasons not to use an autonomous agent for a fixed calculator task.",
    instructions=instructions,
)
experiment_log.append(record)

pd.DataFrame(experiment_log)[
    ["model_requested", "latency_seconds", "input_tokens", "output_tokens", "output"]
]

## 4. Language model application 3.0: Compare instruction variants

Model behavior depends on the instructions and context supplied at inference time. We will keep the task fixed and compare two instruction sets. This is a small controlled experiment: only the instructions should change.

*[Question: If the outputs differ, can we conclude that one instruction set is better from a single example? Why or why not?]*

In [ ]:
task = (
    "A university office processes the same five form types every day. The sequence "
    "of validation steps is known in advance. Should it begin with a deterministic "
    "workflow or an autonomous agent? Explain the trade-off."
)

instruction_variants = {
    "generic": "Answer the question helpfully.",
    "decision_focused": (
        "Act as a systems architect. Recommend the simplest architecture that satisfies "
        "the task. Discuss predictability, flexibility, latency, cost, and failure handling. "
        "State a clear recommendation in the first sentence."
    ),
}

for variant_name, variant_instructions in instruction_variants.items():
    result = run_and_log(task, variant_instructions)
    result["variant"] = variant_name
    experiment_log.append(result)

comparison = pd.DataFrame(experiment_log[-2:])
for _, row in comparison.iterrows():
    print("\nVARIANT:", row["variant"])
    print(row["output"])
    print("Latency:", row["latency_seconds"], "seconds")

## 5. Language model application 4.0: Request a lightweight structured response

Free-form prose is convenient for people but harder for software to validate. Before using formal JSON Schema in Week 2, we will request JSON, parse it, and inspect the result. Prompting for JSON is useful, but it is not equivalent to schema-constrained structured output.

Our software will require three fields: `recommended_architecture`, `reason`, and `needs_human_approval`.

In [ ]:
json_instructions = """
You are a systems-design classifier. Return ONLY valid JSON with exactly these keys:
recommended_architecture: one of [single_call, deterministic_workflow, agent]
reason: a short string
needs_human_approval: a boolean
Do not wrap the JSON in markdown fences.
""".strip()

scenario = (
    "A system must inspect incoming travel reimbursement requests, look up policy, "
    "flag missing evidence, and ask a staff member before rejecting any request."
)

json_record = run_and_log(scenario, json_instructions)
print("Raw model output:")
print(json_record["output"])

try:
    parsed = json.loads(json_record["output"])
    print("\nParsed Python object:")
    print(parsed)
except json.JSONDecodeError as error:
    parsed = None
    print("The response was not valid JSON:", error)

## 6. Workflow 1.0: Add deterministic validation around the model

A workflow combines model output with code-defined control. The model proposes a classification; deterministic Python checks the shape and allowed values. The model does not decide whether validation is optional.

This simple validator illustrates an important course principle: use ordinary software for rules that ordinary software can enforce reliably.

In [ ]:
ALLOWED_ARCHITECTURES = {"single_call", "deterministic_workflow", "agent"}
REQUIRED_KEYS = {"recommended_architecture", "reason", "needs_human_approval"}

def validate_architecture_decision(value):
    errors = []
    if not isinstance(value, dict):
        return ["Output must be a JSON object."]

    missing = REQUIRED_KEYS - set(value)
    extra = set(value) - REQUIRED_KEYS
    if missing:
        errors.append(f"Missing keys: {sorted(missing)}")
    if extra:
        errors.append(f"Unexpected keys: {sorted(extra)}")
    if value.get("recommended_architecture") not in ALLOWED_ARCHITECTURES:
        errors.append("recommended_architecture has an invalid value.")
    if not isinstance(value.get("reason"), str) or not value.get("reason", "").strip():
        errors.append("reason must be a non-empty string.")
    if not isinstance(value.get("needs_human_approval"), bool):
        errors.append("needs_human_approval must be a boolean.")
    return errors

validation_errors = validate_architecture_decision(parsed) if parsed else ["No parsed output."]
if validation_errors:
    print("VALIDATION FAILED")
    for error in validation_errors:
        print("-", error)
else:
    print("VALIDATION PASSED")
    print(parsed)

## 7. Workflow 2.0: Run a small architecture-classification experiment

We will process several scenarios through the same model call and validation step. This is a deterministic workflow because our Python program fixes the sequence: call model → parse → validate → record. The model does not choose new tools or alter the workflow.

In [ ]:
scenarios = [
    "Rewrite a paragraph in plain language.",
    "Validate a known set of form fields and route valid forms to a fixed archive.",
    "Investigate an unfamiliar software failure using logs and several diagnostic tools, adapting the plan after each observation.",
    "Delete duplicate customer records after deciding which record is authoritative.",
]

architecture_results = []
for scenario_text in scenarios:
    record = run_and_log(scenario_text, json_instructions)
    try:
        decision = json.loads(record["output"])
        errors = validate_architecture_decision(decision)
    except json.JSONDecodeError as error:
        decision = {}
        errors = [f"Invalid JSON: {error}"]

    architecture_results.append({
        "scenario": scenario_text,
        "architecture": decision.get("recommended_architecture"),
        "human_approval": decision.get("needs_human_approval"),
        "valid": len(errors) == 0,
        "errors": "; ".join(errors),
        "reason": decision.get("reason"),
        "latency_seconds": record["latency_seconds"],
        "total_tokens": record["total_tokens"],
    })

results_df = pd.DataFrame(architecture_results)
results_df

## 8. Save a sanitized experiment log

Agentic systems need observability, but logs can also leak sensitive information. For this lab, our inputs are synthetic and safe to save. In later labs, we will explicitly decide what should and should not be logged.

The following cell saves the accumulated experiment records. It never saves the API key.

In [ ]:
log_path = "Week1_experiment_log.json"
with open(log_path, "w", encoding="utf-8") as file:
    json.dump(experiment_log, file, indent=2, ensure_ascii=False)

print(f"Saved {len(experiment_log)} experiment records to {log_path}")

## Exercise E1: Extend and critique the architecture classifier

- Add at least six new scenarios: two suited to a single model call, two suited to deterministic workflows, and two that may justify an agent.
- Include at least one high-impact scenario in which human approval is necessary.
- Before calling the model, write your expected classification for every scenario.
- Run the workflow, compare the model's decisions with your expectations, and calculate simple agreement accuracy.
- Identify at least two disagreements or ambiguous cases. Explain whether the model, your expectation, or the three-category taxonomy should be revised.
- Present your analysis in one or more markdown blocks. Do not report accuracy without interpreting the disagreements.

In [ ]:
# EXERCISE E1: Add your scenarios, expected labels, implementation, and evaluation here.
exercise_scenarios = [
    # {"scenario": "...", "expected_architecture": "single_call"},
]

## Exercise E2: Turn a workflow into a bounded agent design—on paper

Choose one scenario from Exercise E1 that may justify an agent. Do **not** implement autonomous tool use yet. Instead:

- Draw or describe a proposed plan–act–observe loop.
- Identify the tools the agent would require and the minimum permissions for each tool.
- Specify the external state or memory that must persist between steps.
- Define at least three stopping conditions.
- Define at least two conditions requiring clarification or human approval.
- List at least three foreseeable failure modes and one mitigation for each.
- Explain why a deterministic workflow would or would not be sufficient.

Write your response in a markdown block below.

### Exercise E2 Response

Replace this text with your design and analysis.